# Semaine 8 — Jour 1 : Évaluation des systèmes IA

Construire une décision de livraison à partir de cas explicites et reproductibles.

Durée indicative : 4 h 30. Prérequis : Python, JSON, tests et notions d’API. Le lab est autonome et ne nécessite ni fournisseur LLM ni clé. Ses sorties sont des fixtures simulées : les résultats valident le protocole, pas les capacités d’un modèle réel.

## Parcours

| Étape | Durée | Résultat |
|---|---:|---|
| Objectifs et cours | 60 min | Contrat d’évaluation |
| Lab et inspection des rapports | 75 min | Deux évaluations comparables |
| Exercices | 45 min | Analyse des erreurs |
| Entretien | 30 min | Argumentation technique |
| Challenge et revue | 60 min | Seuils par catégorie |

Lire `learning_objectives.md`, puis `chapter.md`. Les commandes sont dans `labs/README.md`. Les corrigés se trouvent dans `corriges/`. Les notebooks étudiant et enseignant sont sous `notebooks/week08/`, à la racine du dépôt.

Le jour correspond au premier thème de la semaine 8, « Évaluation ». La roadmap de référence reste inchangée. Ce paquet contient uniquement S8 J1 ; il ne certifie pas l’achèvement des autres semaines.

## Critères de réussite

Expliquer le refus du candidat malgré son meilleur score global, reproduire les rapports, distinguer couverture et qualité, puis ajouter un contrôle par catégorie. Aucun score de ce petit jeu synthétique ne justifie seul une mise en production.


# Objectifs observables

À la fin du jour, vous savez :

1. Définir l’unité évaluée, les entrées, les sorties et le résultat attendu.
2. Construire un jeu versionné avec catégories et cas critiques.
3. Choisir un évaluateur déterministe, humain ou assisté par modèle selon le contrat.
4. Calculer taux global, taux par catégorie et couverture sans supprimer les échecs.
5. Comparer deux versions sur les mêmes cas et repérer les régressions appariées.
6. Appliquer une règle de livraison explicite, indépendante du seul score moyen.
7. Documenter les limites des fixtures et préparer une collecte réelle reproductible.

Preuves attendues : rapports JSON, tests verts, interprétation du cas c09 et solution du challenge.


# Évaluer avant de décider

## 1. Transformer un besoin en contrat

« Le chatbot répond bien » ne décrit aucun test. « Pour une demande de remboursement admissible, il indique le délai de 30 jours et ne modifie aucun dossier sans autorisation » expose deux obligations différentes. Décrire pour chaque cas une entrée, un contexte, une attente et une gravité. Fixer les critères avant d’observer les résultats pour éviter de déplacer le seuil au bénéfice d’une version.

L’unité peut être une réponse, un appel d’outil, une trajectoire d’agent ou l’état final d’un environnement. Une phrase correcte ne prouve pas qu’un outil a été exécuté correctement. Une trajectoire inhabituelle peut cependant parvenir à un état final valide. Évaluer les contraintes de sécurité séparément du succès fonctionnel.

Dans ce lab, les appels d’outils sont des objets à comparer. Ils ne sont jamais exécutés. Tester une autorisation réelle demanderait un environnement isolé et la vérification de ses effets.

## 2. Construire un jeu utile

Commencer par des scénarios représentatifs : tâches fréquentes, ambiguïtés, erreurs d’entrée et comportements interdits. Ajouter les incidents observés après anonymisation et revue. Ne pas mélanger données personnelles de production et fixtures distribuées.

Chaque ligne JSONL contient un identifiant stable, un type, une catégorie, une entrée, une attente et un indicateur critique. Les douze cas du lab couvrent réponses, objets structurés, appels d’outils et refus. Les entrées sont volontairement descriptives : ce sont des tests du moteur, pas des prompts prêts à être soumis à un fournisseur.

Séparer un jeu de développement, utilisé pour améliorer le système, d’un jeu réservé à la validation. Limiter l’accès au second et tracer les changements. Réutiliser continuellement le même jeu pour optimiser les prompts finit par favoriser ses particularités. Une empreinte permet de détecter un changement ; elle ne démontre ni représentativité ni absence de fuite.

## 3. Choisir le bon évaluateur

| Méthode | Utilisation | Limite |
|---|---|---|
| Comparaison exacte | Structure, nom et arguments d’outil | Rejette des variantes équivalentes hors contrat |
| Expressions attendues | Vérification locale de formulations | N’établit ni factualité ni absence de contradiction |
| Revue humaine | Nuance, utilité, cas à fort impact | Coût et désaccord entre annotateurs |
| Modèle évaluateur | Assistance sur des critères sémantiques | Biais, variabilité et sensibilité aux consignes |

Une réponse « ce n’est pas 12 euros » passe le critère lexical « 12 euros ». Ce défaut connu impose une revue ou un critère sémantique pour évaluer la justesse. Le lab conserve ce proxy pour rendre son comportement transparent.

Pour un évaluateur humain, fournir une grille, des exemples et une procédure de désaccord. Pour un modèle évaluateur, conserver sa version et son prompt, calibrer ses décisions sur un échantillon annoté et inspecter faux positifs et faux négatifs. Traiter le contenu évalué comme une donnée susceptible de contenir des instructions malveillantes.

## 4. Scores et couverture

Pour N cas, le taux global est le nombre de réussites divisé par N. Un cas absent reste dans le dénominateur et échoue. La couverture mesure le nombre de sorties présentes divisé par N : une couverture complète ne prouve aucune qualité.

Le taux micro donne le même poids à chaque cas. Le taux macro est la moyenne des taux par catégorie ; une petite catégorie pèse alors autant qu’une grande. Toujours afficher les effectifs pour éviter qu’un score sur deux observations paraisse solide.

Ici, la baseline obtient 10/12 et le candidat 11/12. Le candidat améliore c03 et c04, mais régresse sur c09, un refus inter-tenant critique. La règle accepte une version seulement si la couverture est complète, si le taux atteint 0,8 et si tous les cas critiques passent. La baseline est acceptée par cette règle locale ; le candidat est refusé.

Deux observations de sécurité ne permettent pas d’estimer un taux de risque en production. Ces seuils sont pédagogiques et doivent être justifiés par les conséquences, les exigences et la distribution des tâches avant tout usage réel.

## 5. Comparaison appariée et traçabilité

Comparer les mêmes identifiants avec le même dataset et la même grille. Répertorier améliorations et régressions au lieu de conserver seulement une différence de moyenne. Toute modification du protocole appelle une nouvelle baseline.

Le rapport inclut les empreintes du jeu, des sorties et de la grille. Il ne stocke pas le texte des sorties. Pour une collecte réelle, ajouter commit du système, version du prompt, fournisseur, modèle, paramètres, contexte d’outils, dates et répétitions. Les erreurs d’adaptateur doivent devenir des échecs traçables sans divulguer de secrets.

## 6. Variabilité et décision

Les fixtures donnent toujours le même résultat. Un modèle et ses outils peuvent varier. Répéter les cas, conserver chaque essai, analyser la dispersion et les erreurs par segment. Ne pas sélectionner uniquement le meilleur essai. Une évolution du fournisseur ou du corpus exige une nouvelle mesure.

Une évaluation avant livraison et l’observation en production se complètent. La première teste un contrat sur un jeu connu ; la seconde détecte des situations et distributions nouvelles. Un incident peut alimenter un nouveau cas après validation, sans remplacer le jeu réservé par les seuls incidents.

## 7. Procédure de revue

1. Vérifier l’intégrité des entrées et la couverture.
2. Inspecter les cas critiques et les erreurs par catégorie.
3. Comparer les versions sous le même protocole.
4. Appliquer les seuils et documenter les exceptions autorisées par le responsable.
5. Enregistrer une décision avec limites, preuves et date.

Le rapport du lab automatise une règle locale ; la responsabilité de livraison reste une décision située. Une bonne moyenne ne neutralise pas une violation critique.


# Lab exécutable

Les cellules lisent les fixtures distribuées. Elles ne contactent aucun modèle.

In [ ]:
from pathlib import Path
import sys
import json
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "book/week08/day01/labs/evaluation_lab").is_dir()), None)
assert root is not None, "Ouvrir le notebook dans le dépôt"
day = root / "book/week08/day01"
sys.path.insert(0, str(day / "labs"))
from evaluation_lab.core import read_rows, evaluate, compare
dataset = read_rows(day / "assets/dataset.jsonl")
baseline = evaluate(dataset, read_rows(day / "assets/baseline.jsonl"))
candidate = evaluate(dataset, read_rows(day / "assets/candidate.jsonl"))
print(json.dumps({"baseline": baseline["rate"], "candidate": candidate["rate"], "comparison": compare(baseline, candidate)}, indent=2))
assert baseline["accepted"] and not candidate["accepted"]

Inspecter la distribution des réussites et les effectifs.

In [ ]:
for name, report in (("baseline", baseline), ("candidate", candidate)):
    print(name, report["segments"], "coverage:", report["coverage"])
assert compare(baseline, candidate)["regressions"] == ["c09"]

# Exercices

1. Calculer à la main les taux globaux des deux versions et leur différence en points de pourcentage. Identifier les changements appariés.
2. Supprimer c12 des prédictions baseline. Prévoir couverture, qualité et décision, puis vérifier avec `evaluate`.
3. Tester « ce n’est pas 12 euros » sur c04. Expliquer le faux positif et proposer un critère plus adapté.
4. Comparer `{ "x": true }` et `{ "x": 1 }`. Pourquoi une égalité Python ordinaire ne suffit-elle pas au contrat strict ?
5. Proposer quatre cas nouveaux : deux représentatifs, un adversarial et un cas d’erreur d’outil. Justifier catégorie, attente et gravité.
6. Préparer le contrat d’un collecteur réel : métadonnées, erreurs, répétitions et confidentialité. Expliquer ce que les empreintes ne garantissent pas.


# Entretien technique

1. Qu’évaluez-vous dans un agent : sa réponse, ses appels ou son résultat ?
2. Comment justifier une livraison refusée malgré une moyenne meilleure ?
3. Quelle différence entre couverture et taux de réussite ?
4. Quand employer un modèle évaluateur et comment le contrôler ?
5. Comment comparer des systèmes non déterministes ?
6. Pourquoi un jeu réservé à la validation se dégrade-t-il si l’équipe l’utilise pour optimiser ?
7. Que doit conserver un rapport pour rester interprétable dans trois mois ?
8. Quelle différence entre une évaluation locale acceptée et une preuve de sûreté en production ?


# Challenge : ajouter des seuils par catégorie

Implémenter `segment_gate(report, thresholds)` sans modifier la règle existante. Retourner un booléen. Exiger que la règle globale passe et que chaque catégorie demandée existe et atteigne son seuil. Rejeter une configuration vide, un seuil non numérique, un booléen, NaN ou un seuil hors de [0, 1].

Démontrer : baseline avec support à 0,6 acceptée ; baseline avec support à 0,8 refusée ; candidat toujours refusé malgré ses réponses corrigées ; catégorie absente refusée. Ajouter au moins trois assertions de configuration invalide.

Livrables : fonction, assertions, justification des seuils et limites du petit échantillon. Ne pas remplacer le jeu ni changer les attentes pour réussir le challenge.


# Corrigé des exercices

1. Baseline : 83,33 % ; candidat : 91,67 % ; différence : +8,33 points. c03 et c04 progressent, c09 régresse. c09 suffit à refuser le candidat.
2. c12 réussissait. La couverture tombe à 11/12, le taux baseline à 9/12. La décision est refusée. Le dénominateur demeure douze.
3. Le proxy lexical passe puisque l’expression apparaît. Il faut une grille vérifiant le montant affirmé et l’absence de contradiction, calibrée sur des exemples positifs et négatifs. Une réponse structurée peut aider lorsque le produit autorise ce format.
4. Python considère `True == 1` vrai. Le moteur compare des sérialisations JSON canoniques qui préservent les types. Tester aussi les clés supplémentaires, les listes et `1.0`.
5. Exemple : délai d’une zone différente, demande de remboursement hors délai, tentative d’accès au dossier d’un autre tenant, indisponibilité de l’outil de commande. Les attentes doivent indiquer les faits ou comportements autorisés ; le dernier cas attend une erreur contrôlée et aucune invention de statut.
6. Tracer commit, prompts, modèle, paramètres, corpus, outils et essais. Conserver une erreur par cas après timeout et limiter les traces sensibles. Les empreintes détectent un changement du contenu suivi ; elles ne prouvent pas que le modèle a réellement utilisé ces paramètres, ni que le jeu est représentatif.


# Réponses attendues

1. Partir du contrat. Évaluer résultat final et contraintes ; inspecter appels et trajectoire quand ils expliquent les erreurs ou violations. Une seule trajectoire exacte peut surcontraindre une solution valide.
2. Montrer la régression critique appariée et la règle fixée avant l’expérience. Le score global compense numériquement des erreurs dont les conséquences diffèrent.
3. La couverture indique les sorties disponibles ; la réussite indique les sorties conformes. Un timeout ne doit pas disparaître du dénominateur.
4. Pour des propriétés sémantiques difficiles à coder. Calibrer contre une revue humaine, vérifier désaccords et biais, versionner la grille et isoler les instructions du contenu évalué.
5. Répéter avec un protocole fixe, conserver tous les essais, examiner dispersion et segments. Décrire l’incertitude ; un seul passage ne prouve pas une amélioration stable.
6. L’optimisation adapte le système aux exemples connus. Le jeu perd sa fonction de validation indépendante ; réserver un autre jeu et organiser ses accès.
7. Jeu, grille, sorties ou empreintes avec accès aux artefacts, versions système et collecte, paramètres, effectifs, erreurs, décision et limites.
8. Le jeu local n’échantillonne pas toutes les situations réelles. Une preuve de réussite sur douze fixtures ne caractérise pas les risques d’un déploiement.


# Solution du challenge

Cette fonction ajoute des exigences ; elle conserve la décision globale et les refus critiques. Une catégorie absente échoue plutôt que d’être ignorée.

In [ ]:
import math

def segment_gate(report, thresholds):
    if not isinstance(thresholds, dict) or not thresholds:
        raise ValueError("thresholds required")
    for category, threshold in thresholds.items():
        if not isinstance(category, str) or not category.strip():
            raise ValueError("invalid category")
        if type(threshold) not in (int, float) or not math.isfinite(threshold) or not 0 <= threshold <= 1:
            raise ValueError("invalid threshold")
    return report["accepted"] and all(
        category in report["segments"] and report["segments"][category]["rate"] >= threshold
        for category, threshold in thresholds.items()
    )

assert segment_gate(baseline, {"support": 0.6})
assert not segment_gate(baseline, {"support": 0.8})
assert not segment_gate(candidate, {"support": 0.8})
assert not segment_gate(baseline, {"unknown": 0.5})
for invalid in ({}, {"support": True}, {"support": float("nan")}, {"support": 1.1}):
    try:
        segment_gate(baseline, invalid)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid configuration accepted")
print("Challenge : 8 assertions validées")

Le groupe support contient cinq cas. Trois réussissent dans la baseline, soit 0,6. Le seuil de 0,8 demande quatre réussites. Ce seuil illustratif doit être justifié par le besoin produit ; il n’est pas une estimation fiable du taux réel sur seulement cinq cas.

# Revue enseignant

| Critère | Points |
|---|---:|
| Contrat et limites du proxy lexical | 4 |
| Calculs, couverture et comparaison appariée | 4 |
| Analyse de c09 et règle critique | 4 |
| Challenge, configuration invalide et assertions | 5 |
| Traçabilité et préparation d’une collecte réelle | 3 |

Réussite indicative : 14/20 avec compréhension obligatoire de c09. Une réponse disant « le candidat est meilleur donc livrable » ne satisfait pas le critère critique.

Points à vérifier : les fixtures ne proviennent pas d’un LLM ; aucun outil métier n’est exécuté ; les sorties manquantes comptent comme échecs ; les clés supplémentaires échouent ; les taux sont accompagnés d’effectifs ; les corrections restent absentes du notebook étudiant.

## Améliorations possibles

Ajouter des scénarios réels anonymisés, une collecte avec erreurs contrôlées, une revue humaine calibrée, des répétitions et un jeu réservé. Ces pistes ne modifient pas la roadmap ou les spécifications figées.


# Références

Sources primaires consultées le 2 octobre 2026. Le cours propose une synthèse originale et un lab local ; il ne reproduit pas les exemples de ces publications.

- Anthropic, [Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents), 9 janvier 2026 : distinguer évaluateurs, résultats et trajectoires.
- NIST, [AI Measurement and Evaluation](https://www.nist.gov/ai-measurement-and-evaluation) : évaluation et mesure des systèmes IA.
- NIST, [AI RMF — Core](https://airc.nist.gov/airmf-resources/airmf/5-sec-core/) : démarche de mesure liée aux risques.
- NIST, [Generative AI Profile](https://www.nist.gov/publications/artificial-intelligence-risk-management-framework-generative-artificial-intelligence), 2024 : contexte de gestion des risques de l’IA générative.
- Python, [json](https://docs.python.org/3/library/json.html) et [unittest](https://docs.python.org/3/library/unittest.html) : bibliothèques utilisées dans le lab.
